In [1]:
# ================================================================
# CELLULE 1 - INSTALLATION ET GOOGLE DRIVE
# ================================================================

!pip install -q \
    langchain \
    langchain-community \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes \
    faiss-cpu \
    pypdf \
    torch \
    gradio



In [7]:
!pip install --upgrade langchain langchain-community
!pip install --upgrade numpy torch transformers sentence-transformers faiss-cpu


  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.7 MB/s eta 0:00:00
  Using cached langchain_text_splitters-1.1.0-py3-none-any.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 9.1 MB/s eta 0:00:00
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.4/476.4 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.7/273.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 120.7 MB/s eta 0:00:00
Using cached langchain_text_splitters-1.1.0-py3-none-any.whl (34 kB)
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.147
    Uninstalling langsmith-0.1.147:
      Successfully uninstalled langsmith-0.1.147
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.1.53
    Uninstalling langchain-core-0.1.53:
   

In [1]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/RAG/data "
import os
os.makedirs(DATA_PATH, exist_ok=True)

print("Drive monté et dépendances installées")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive monté et dépendances installées


In [5]:
pip install -U langchain-text-splitters


In [2]:
# ================================================================
# CELLULE 2 - LLM, EMBEDDINGS ET DOCUMENTS (LANGCHAIN)
# ================================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline


# ---------------------
# DEVICE
# ---------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé: {device}")

# ---------------------
# SOLUTION STABLE : Phi-2 (2.7B) - Excellent pour Colab
# ---------------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gc

# Nettoyer d'abord la mémoire
gc.collect()
torch.cuda.empty_cache()

# Phi-2 de Microsoft - Très performant et stable
model_name = "microsoft/phi-2"

print(f" Chargement du modèle {model_name}...")


tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Chargement simple et stable
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

# Déplacer sur GPU si disponible
if torch.cuda.is_available():
    model = model.to("cuda")
    print(" Modèle chargé sur GPU")
else:
    print(" Modèle chargé sur CPU")

print(" Configuration du pipeline...")
generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
    top_p=0.85,
    top_k=50,
    repetition_penalty=1.2,
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.pad_token_id
)

llm = HuggingFacePipeline(pipeline=generation_pipeline)

print(f" Phi-2 chargé avec succès !")
print(f" Taille du modèle : 2.7B paramètres")
print(f" Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.2f} GB" if torch.cuda.is_available() else "")

# ---------------------
# EMBEDDINGS
# ---------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

print("Embeddings chargés")

# ---------------------
# CHARGEMENT DES PDFS
# ---------------------
loader = PyPDFDirectoryLoader(DATA_PATH)
documents = loader.load()

print(f"{len(documents)} documents chargés")

# ---------------------
# SPLITTING
# ---------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

splits = text_splitter.split_documents(documents)

print(f"{len(splits)} chunks créés")


Device utilisé: cuda
 Chargement du modèle microsoft/phi-2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


 Modèle chargé sur GPU
 Configuration du pipeline...
 Phi-2 chargé avec succès !
 Taille du modèle : 2.7B paramètres
 Mémoire GPU utilisée : 5.56 GB


/tmp/ipython-input-1091797649.py:76: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=generation_pipeline)
/tmp/ipython-input-1091797649.py:85: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Embeddings chargés
235 documents chargés
246 chunks créés


In [24]:
!ls "/content/drive/MyDrive/RAG/data "

'CH2_Big Data Analytics.pdf'  'formation power BI perf.pdf'
'CH3_Big Data Analytics.pdf'   Présentation_SMA_JADE.pdf
 chap3.pdf


In [3]:
import langchain
import langchain_community
import numpy
import torch

print("LangChain version:", langchain.__version__)
print("LangChain-Community version:", langchain_community.__version__)
print("NumPy version:", numpy.__version__)
print("Torch version:", torch.__version__)


LangChain version: 1.2.0
LangChain-Community version: 0.4.1
NumPy version: 2.3.5
Torch version: 2.9.1+cu128


In [3]:
# ================================================================
# CELLULE 3 - RAG LANGCHAIN + INTERFACE (CORRIGÉE - V3)
# ================================================================

import gradio as gr

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores.faiss import FAISS

# ---------------------
# VECTOR DATABASE (FAISS)
# ---------------------
vectorstore = FAISS.from_documents(
    splits,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("FAISS index créé")

# ---------------------
# RAG CHAIN (CORRIGÉE)
# ---------------------

# Définir le prompt EN FRANÇAIS
template = """
Tu es un assistant expert qui répond aux questions en te basant sur les documents fournis.
Utilise les informations du contexte ci-dessous pour répondre à la question.
Si tu ne trouves pas la réponse dans le contexte, dis-le clairement.
Réponds de manière concise et précise, en français uniquement.

Contexte : {context}

Question : {question}

Réponse détaillée en français :"""

prompt = ChatPromptTemplate.from_template(template)

# Fonction pour formater les documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Fonction wrapper pour appeler le LLM
def call_llm(prompt_value):
    """
    Appelle le LLM et retourne une chaîne de caractères propre
    """
    # Convertir le prompt en string
    if hasattr(prompt_value, 'to_string'):
        prompt_str = prompt_value.to_string()
    elif hasattr(prompt_value, 'text'):
        prompt_str = prompt_value.text
    else:
        prompt_str = str(prompt_value)

    # Appel au LLM
    response = llm.invoke(prompt_str)

    # Extraction de la réponse selon le format retourné
    if isinstance(response, str):
        return response
    elif isinstance(response, dict):
        for key in ['generated_text', 'text', 'content']:
            if key in response:
                return response[key]
        return str(list(response.values())[0]) if response else ""
    elif isinstance(response, list) and len(response) > 0:
        first = response[0]
        if isinstance(first, dict):
            for key in ['generated_text', 'text', 'content']:
                if key in first:
                    return first[key]
            return str(first)
        return str(first)
    elif hasattr(response, 'content'):
        return response.content
    else:
        return str(response)

# *** CORRECTION : Utiliser invoke au lieu de get_relevant_documents ***
def retrieve_docs(inputs):
    """
    Extrait la question et récupère les documents pertinents
    """
    # Si inputs est un dict avec une clé 'question'
    if isinstance(inputs, dict) and 'question' in inputs:
        query = inputs['question']
    else:
        query = str(inputs)

    # Utiliser invoke (nouvelle API) au lieu de get_relevant_documents
    docs = retriever.invoke(query)
    return docs

# Construire la chaîne RAG avec extraction explicite de la question
rag_chain = (
    {
        "context": RunnableLambda(retrieve_docs) | format_docs,
        "question": lambda x: x["question"] if isinstance(x, dict) else x
    }
    | prompt
    | call_llm
)

print("RAG chain prête")

# ---------------------
# CHAT FUNCTION
# ---------------------
def chat_interface(query):
    if not query.strip():
        return "Veuillez poser une question."

    try:
        # Appeler la chaîne avec la requête
        answer = rag_chain.invoke({"question": query})

        # Nettoyer la réponse (enlever le prompt si présent)
        if isinstance(answer, str):
            # Chercher différents marqueurs possibles
            for marker in ["Réponse détaillée en français :", "Réponse :", "Answer:"]:
                if marker in answer:
                    answer = answer.split(marker)[-1].strip()
                    break
            # Enlever la question si elle est répétée
            if query in answer:
                answer = answer.replace(query, "").strip()

        # Récupérer les documents sources séparément (avec invoke)
        docs = retriever.invoke(query)

        sources_text = "\n\n### 📚 Sources utilisées\n"
        seen_sources = set()

        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get('source', 'Document inconnu')
            # Extraire juste le nom du fichier
            if '/' in source:
                source = source.split('/')[-1]

            # Éviter les doublons
            if source not in seen_sources:
                seen_sources.add(source)
                page = doc.metadata.get('page', 'N/A')
                sources_text += f"{len(seen_sources)}. **{source}** (page {page})\n"

        return f"### 💬 Réponse\n\n{answer}\n{sources_text}"

    except Exception as e:
        print(f"Erreur détaillée : {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        return f"❌ **Une erreur est survenue**\n\n```\n{str(e)}\n```"

# ---------------------
# GRADIO INTERFACE
# ---------------------
with gr.Blocks(title="RAG LangChain avec Mistral-7B", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🤖 Assistant RAG Intelligent")
    gr.Markdown("### 💡 Posez vos questions sur vos documents PDF (Power BI, Big Data, JADE...)")

    with gr.Row():
        with gr.Column(scale=2):
            question = gr.Textbox(
                label="❓ Votre question",
                placeholder="Exemple : Quelle est la différence entre SUM et SUMX en DAX ?",
                lines=4
            )

            with gr.Row():
                submit = gr.Button("🚀 Générer la réponse", variant="primary", size="lg")
                clear = gr.Button("🗑️ Effacer", size="lg")

            gr.Markdown("**Exemples de questions :**")
            gr.Examples(
                examples=[
                    "Quelle est la différence entre SUM et SUMX en DAX ?",
                    "Qu'est-ce que Big Data Analytics ?",
                    "Explique-moi le framework JADE",
                    "Comment fonctionne Power BI ?",
                ],
                inputs=question
            )

    with gr.Row():
        output = gr.Markdown(label="📝 Réponse")

    submit.click(
        fn=chat_interface,
        inputs=question,
        outputs=output
    )

    clear.click(
        fn=lambda: ("", ""),
        inputs=None,
        outputs=[question, output]
    )

print("✅ Interface Gradio prête")
app.launch(debug=True, share=True)

FAISS index créé
RAG chain prête


/tmp/ipython-input-2780766016.py:164: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="RAG LangChain avec Mistral-7B", theme=gr.themes.Soft()) as app:


✅ Interface Gradio prête
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://497b9a35c1b1342af2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://497b9a35c1b1342af2.gradio.live


In [30]:
!pip install -q langchain-huggingface

In [8]:
# Vérification des variables nécessaires
print("Variables présentes :")
print("- llm :", 'llm' in locals() or 'llm' in globals())
print("- retriever :", 'retriever' in locals() or 'retriever' in globals())
print("- rag_chain_with_source :", 'rag_chain_with_source' in locals() or 'rag_chain_with_source' in globals())
print("- vectorstore :", 'vectorstore' in locals() or 'vectorstore' in globals())

Variables présentes :
- llm : True
- retriever : True
- rag_chain_with_source : False
- vectorstore : True


In [7]:
# ================================================================
# CELLULE DE VALIDATION RAG AVEC RAGAs (CORRIGÉE)
# ================================================================

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Assurez-vous que ragas est installé
# !pip install -q ragas

from datasets import Dataset
import pandas as pd

# Imports Ragas
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas import evaluate

# Imports LangChain (normalement déjà faits)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS

# 1. Générer un petit jeu de données de test
print("--- Préparation des données de test pour RAGAs ---")

# Exemple de questions simples basées sur vos documents
sample_docs = splits[:5] # Prendre les 5 premiers chunks comme exemple
test_data = []
for doc in sample_docs:
    content_snippet = doc.page_content[:100].strip() # Premier 100 caractères
    question = f"Quel est le sujet principal de ce passage : '{content_snippet}...' ?"
    test_data.append({
        "question": question,
        "ground_truth_context": [doc.page_content], # Contexte attendu
        "ground_truth_answer": "Non spécifié", # Réponse de référence (optionnelle pour certaines métriques)
        "source_document": doc.metadata.get("source", "Unknown")
    })

# Conversion en DataFrame Pandas
df_test = pd.DataFrame(test_data)
print(f"Jeu de test généré avec {len(df_test)} questions.")

# Pour RAGAs, on a besoin d'un Dataset HuggingFace
dataset_ragas = Dataset.from_pandas(df_test)

# 2. Définir la chaîne RAG pour RAGAs (CORRIGÉE)
# On adapte la chaîne précédente pour qu'elle renvoie les bons éléments
# pour l'évaluation RAGAs : answer, contexts, retrieved_contexts

def get_rag_answer_and_contexts(inputs):
    question = inputs["question"]
    # Récupérer les documents en utilisant .invoke() au lieu de .get_relevant_documents()
    # La méthode .invoke() est la façon standard de LangChain pour interagir avec un Runnable
    # Le retriever est un Runnable
    retrieved_docs = retriever.invoke(question) # <- CORRECTION ICI
    contexts = [doc.page_content for doc in retrieved_docs]
    # Générer la réponse
    answer = rag_chain_with_source.invoke({"question": question})
    # Retourner un dictionnaire avec les clés attendues par RAGAs
    return {
        "answer": answer,
        "contexts": contexts,
        "retrieved_contexts": contexts # Pour RAGAs, souvent identique à contexts
    }

# Appliquer la chaîne à chaque exemple du dataset
print("--- Génération des réponses et contextes pour RAGAs ---")
# On mappe la fonction sur le dataset
try:
    result_dataset = dataset_ragas.map(get_rag_answer_and_contexts)
except AttributeError as e:
    print(f"Erreur persistante avec le retriever : {e}")
    print("Vérifiez la documentation de votre version de LangChain pour la méthode correcte.")
    # En dernier recours, on peut tenter de forcer la récupération via la vectorstore directement
    print("Tentative alternative via la méthode directe de la vectorstore...")
    def get_rag_answer_and_contexts_alt(inputs):
        question = inputs["question"]
        # Utilisation directe de la vectorstore avec similarity_search
        retrieved_docs = vectorstore.similarity_search(question, k=5) # Utilise la variable 'vectorstore' créée dans la cellule 3
        contexts = [doc.page_content for doc in retrieved_docs]
        answer = rag_chain_with_source.invoke({"question": question})
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieved_contexts": contexts
        }
    result_dataset = dataset_ragas.map(get_rag_answer_and_contexts_alt) # Utilise la fonction alternative
    print("Utilisation de la méthode alternative réussie.")


# 3. Définir les métriques RAGAs à utiliser
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    # context_recall, # Peut nécessiter des 'ground_truth_contexts' dans le dataset final
]

print("--- Lancement de l'évaluation RAGAs ---")
try:
    score = evaluate(
        dataset=result_dataset,
        metrics=metrics,
    )
    print("\n--- Résultats RAGAs ---")
    print(score)
    print("\nDataFrame des scores :")
    df_scores = score.to_pandas()
    print(df_scores)

    print("\n--- Évaluation RAGAs terminée ---")

except Exception as e:
    print(f"❌ Erreur lors de l'évaluation RAGAs : {e}")
    import traceback
    traceback.print_exc()


--- Préparation des données de test pour RAGAs ---
Jeu de test généré avec 5 questions.
--- Génération des réponses et contextes pour RAGAs ---


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

NameError: name 'rag_chain_with_source' is not defined

In [5]:
!pip install -q ragas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.9/419.9 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 26.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s